# MSCNet Baseline Training (Phase 0)

**AGMS-Net MTP Project** — Reproducing MSCNet (IEEE TSC 2025) on Alibaba Cluster Trace 2018

This notebook:
1. Clones the project repo
2. Installs dependencies
3. Uploads the preprocessed `alibaba.csv` dataset
4. Trains MSCNet for all 4 prediction horizons (H=24, 48, 72, 96)
5. Reports MSE/MAE/RMSE and compares against paper values

**Runtime**: Make sure you select **GPU** runtime (Runtime > Change runtime type > T4 GPU)

## 1. Setup: Clone Repo & Install Dependencies

In [ ]:
# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone the project repo
!git clone https://github.com/patelkanak23/Workload_Prediction_MTP.git
%cd Workload_Prediction_MTP

In [ ]:
# Install missing dependencies (Colab has torch, numpy, pandas, sklearn pre-installed)
!pip install -q einops ptflops torchinfo reformer-pytorch PyWavelets sktime patool matplotlib tqdm

## 2. Fix Known Issues in MSCNet Baseline Code

The official MSCNet repo has a few minor issues that need patching:

In [ ]:
%%writefile MSCNet_baseline/exp/exp_basic.py
import os
import torch
from models import MSCNet

# MSCNet_base doesn't exist in the repo - removed the broken import
# Also adding ablation variants so they can be used if needed
from models import MSCNet_conv, MSCNet_patch, MSCNet_trend


class Exp_Basic(object):
    def __init__(self, args):
        self.args = args
        self.model_dict = {
            'MSCNet': MSCNet,
            'MSCNet_conv': MSCNet_conv,
            'MSCNet_patch': MSCNet_patch,
            'MSCNet_trend': MSCNet_trend,
        }
        self.device = self._acquire_device()
        self.model = self._build_model().to(self.device)

    def _build_model(self):
        raise NotImplementedError
        return None

    def _acquire_device(self):
        if self.args.use_gpu:
            os.environ["CUDA_VISIBLE_DEVICES"] = str(
                self.args.gpu) if not self.args.use_multi_gpu else self.args.devices
            device = torch.device('cuda:{}'.format(self.args.gpu))
            print('Use GPU: cuda:{}'.format(self.args.gpu))
        else:
            device = torch.device('cpu')
            print('Use CPU')
        return device

    def _get_data(self):
        pass

    def vali(self):
        pass

    def train(self):
        pass

    def test(self):
        pass

## 3. Upload Dataset

Upload `alibaba.csv` from your local machine. The file is ~1.8 MB.

In [ ]:
import os

# Create dataset directory inside MSCNet_baseline (where run.py expects it)
os.makedirs('MSCNet_baseline/dataset', exist_ok=True)

# Check if dataset already exists (e.g., from Google Drive)
if os.path.exists('MSCNet_baseline/dataset/alibaba.csv'):
    print('Dataset already exists!')
elif os.path.exists('dataset/alibaba.csv'):
    # Copy from repo's dataset dir if it was committed
    import shutil
    shutil.copy('dataset/alibaba.csv', 'MSCNet_baseline/dataset/alibaba.csv')
    print('Copied from repo dataset/')
else:
    # Upload from local machine
    from google.colab import files
    print('Please upload alibaba.csv (1.8 MB):')
    uploaded = files.upload()
    
    # Move to the right place
    for filename in uploaded:
        os.rename(filename, 'MSCNet_baseline/dataset/alibaba.csv')
        print(f'Moved {filename} to MSCNet_baseline/dataset/alibaba.csv')

# Verify
import pandas as pd
df = pd.read_csv('MSCNet_baseline/dataset/alibaba.csv')
print(f'\nDataset loaded: {df.shape}')
print(f'Columns: {list(df.columns[:5])} ... {list(df.columns[-3:])}')
print(f'Time range: {df["date"].iloc[0]} to {df["date"].iloc[-1]}')

## 4. Train MSCNet — All 4 Horizons

Using the exact hyperparameters from `scripts/alibaba.sh`:
- `seq_len=96`, `label_len=96`
- `features=M`, `enc_in=100`
- `e_layers=3`, `n_heads=4`, `d_model=128`
- `train_epochs=100`, `patience=3`

In [ ]:
%cd /content/Workload_Prediction_MTP/MSCNet_baseline

import os
os.makedirs('logs/long_term_forecast_workload', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('Working directory:', os.getcwd())
print('Dataset exists:', os.path.exists('dataset/alibaba.csv'))
print('Ready to train!')

In [ ]:
# Train for pred_len = 24
!python -u run.py \
    --task_name long_term_forecast \
    --is_training 1 \
    --root_path ./dataset \
    --data_path alibaba.csv \
    --model_id alibaba_96_24 \
    --model MSCNet \
    --data custom \
    --features M \
    --freq t \
    --seq_len 96 \
    --label_len 96 \
    --pred_len 24 \
    --e_layers 3 \
    --d_layers 1 \
    --factor 3 \
    --enc_in 100 \
    --dec_in 100 \
    --c_out 100 \
    --des Exp \
    --itr 1 \
    --n_heads 4 \
    --d_model 128 \
    --train_epochs 100 \
    --patience 3

In [ ]:
# Train for pred_len = 48
!python -u run.py \
    --task_name long_term_forecast \
    --is_training 1 \
    --root_path ./dataset \
    --data_path alibaba.csv \
    --model_id alibaba_96_48 \
    --model MSCNet \
    --data custom \
    --features M \
    --freq t \
    --seq_len 96 \
    --label_len 96 \
    --pred_len 48 \
    --e_layers 3 \
    --d_layers 1 \
    --factor 3 \
    --enc_in 100 \
    --dec_in 100 \
    --c_out 100 \
    --des Exp \
    --itr 1 \
    --n_heads 4 \
    --d_model 128 \
    --train_epochs 100 \
    --patience 3

In [ ]:
# Train for pred_len = 72
!python -u run.py \
    --task_name long_term_forecast \
    --is_training 1 \
    --root_path ./dataset \
    --data_path alibaba.csv \
    --model_id alibaba_96_72 \
    --model MSCNet \
    --data custom \
    --features M \
    --freq t \
    --seq_len 96 \
    --label_len 96 \
    --pred_len 72 \
    --e_layers 3 \
    --d_layers 1 \
    --factor 3 \
    --enc_in 100 \
    --dec_in 100 \
    --c_out 100 \
    --des Exp \
    --itr 1 \
    --n_heads 4 \
    --d_model 128 \
    --train_epochs 100 \
    --patience 3

In [ ]:
# Train for pred_len = 96
!python -u run.py \
    --task_name long_term_forecast \
    --is_training 1 \
    --root_path ./dataset \
    --data_path alibaba.csv \
    --model_id alibaba_96_96 \
    --model MSCNet \
    --data custom \
    --features M \
    --freq t \
    --seq_len 96 \
    --label_len 96 \
    --pred_len 96 \
    --e_layers 3 \
    --d_layers 1 \
    --factor 3 \
    --enc_in 100 \
    --dec_in 100 \
    --c_out 100 \
    --des Exp \
    --itr 1 \
    --n_heads 4 \
    --d_model 128 \
    --train_epochs 100 \
    --patience 3

## 5. Results Summary

Collect and compare against paper-reported values.

In [ ]:
import numpy as np
import os
import glob

# Paper reference values (Alibaba, L=96)
paper_values = {
    24: {'MSE': 0.0030, 'MAE': 0.0292, 'RMSE': 0.0544},
    48: {'MSE': 0.0041, 'MAE': 0.0356, 'RMSE': 0.0637},
    72: {'MSE': 0.0050, 'MAE': 0.0399, 'RMSE': 0.0707},
    96: {'MSE': 0.0058, 'MAE': 0.0433, 'RMSE': 0.0761},
}

print('=' * 80)
print('MSCNet Reproduction Results vs. Paper (Alibaba, L=96)')
print('=' * 80)
print(f'{"H":>4} | {"Our MSE":>10} | {"Paper MSE":>10} | {"Delta%":>7} | {"Our MAE":>10} | {"Paper MAE":>10} | {"Delta%":>7}')
print('-' * 80)

# Find result directories
results_dirs = sorted(glob.glob('results/*alibaba*'))

our_results = {}
for d in results_dirs:
    metrics_path = os.path.join(d, 'metrics.npy')
    if os.path.exists(metrics_path):
        # metrics.npy contains [mae, mse, rmse, mape, mspe, ms]
        metrics = np.load(metrics_path)
        mae, mse, rmse = metrics[0], metrics[1], metrics[2]
        
        # Extract pred_len from directory name
        for h in [24, 48, 72, 96]:
            if f'_pl{h}_' in d or f'_{h}_' in d:
                our_results[h] = {'MSE': mse, 'MAE': mae, 'RMSE': rmse}
                paper = paper_values[h]
                mse_delta = (mse - paper['MSE']) / paper['MSE'] * 100
                mae_delta = (mae - paper['MAE']) / paper['MAE'] * 100
                print(f'{h:>4} | {mse:>10.6f} | {paper["MSE"]:>10.6f} | {mse_delta:>+6.1f}% | {mae:>10.6f} | {paper["MAE"]:>10.6f} | {mae_delta:>+6.1f}%')
                break

print('=' * 80)

if not our_results:
    print('\nNo results found. Check that training completed successfully.')
    print('Looking for results in:', os.getcwd())
    print('Dirs found:', results_dirs)
    # Try alternate location
    alt_results = glob.glob('./results/*/metrics.npy')
    print('Metrics files:', alt_results)
else:
    print(f'\nNote: Deviations within +/-15% are normal for reproductions.')
    print('Differences arise from: random seed effects, data preprocessing')
    print('variations, hardware differences, and potential minor code updates.')

In [ ]:
# Also print the raw result log if it exists
if os.path.exists('result_long_term_forecast.txt'):
    print('\n--- Raw Results Log ---')
    with open('result_long_term_forecast.txt', 'r') as f:
        print(f.read())

## 6. Save Results to Google Drive (Optional)

Save checkpoints and results to Google Drive so you can access them locally.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Create save directory on Drive
drive_save_dir = '/content/drive/MyDrive/AGMS_Net_Results/Phase0'
os.makedirs(drive_save_dir, exist_ok=True)

# Copy results
if os.path.exists('results'):
    shutil.copytree('results', f'{drive_save_dir}/results', dirs_exist_ok=True)
    print(f'Results saved to {drive_save_dir}/results')

# Copy checkpoints
if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', f'{drive_save_dir}/checkpoints', dirs_exist_ok=True)
    print(f'Checkpoints saved to {drive_save_dir}/checkpoints')

# Copy result log
if os.path.exists('result_long_term_forecast.txt'):
    shutil.copy('result_long_term_forecast.txt', f'{drive_save_dir}/result_long_term_forecast.txt')
    print(f'Result log saved to {drive_save_dir}/')

print('\nAll results saved to Google Drive!')